In [ ]:
import pandas as pd
import numpy as np
from snowflake.snowpark.context import get_active_session

# No Snowpark Worksheets, use get_active_session()
session = get_active_session()

sql_query = """
SELECT *
FROM (
    SELECT
        ID_CONTRACT,
        SEGMENT,
        DOWN_NOMINAL_BROADBAND_MBPS AS SPEED,
        VOL_DL_MONTH_GB AS CONSUMO,
        SOURCE_FILE,
        DAT_CRG_RGT_SNW,
        COALESCE(
            TO_DATE(REGEXP_SUBSTR(SOURCE_FILE, '[0-9]{8}'), 'YYYYMMDD'),
            TO_DATE(REGEXP_SUBSTR(SOURCE_FILE, '[0-9]{4}-[0-9]{2}-[0-9]{2}'), 'YYYY-MM-DD')
        ) AS DATA
    FROM dwdev.DWADM.IND_BEEGOL_WEEKLY
)
QUALIFY DAT_CRG_RGT_SNW = MAX(DAT_CRG_RGT_SNW)
       OVER (PARTITION BY SOURCE_FILE)
"""

df_raw = session.sql(sql_query)
                                
df_raw_pd = df_raw.to_pandas()

df_raw_pd['SEMA_REF'] = df_raw_pd['SOURCE_FILE'].str.extract(r'/(\d{8})_')[0]
df_raw_pd['SEMA_REF'] = pd.to_datetime(df_raw_pd['SEMA_REF'], format='%Y%m%d', errors='coerce')

df_raw_emp = df_raw_pd.drop(columns=['SOURCE_FILE'])

df_raw_emp = df_raw_emp.dropna(subset=['CONSUMO'])

agg_dict = {
    "CONSUMO": [
        'count',
        'mean',
        'median',
        'min',
        'max',
        lambda x: np.percentile(x.dropna(), 70),
        lambda x: np.percentile(x.dropna(), 75),
        lambda x: np.percentile(x.dropna(), 80),
        lambda x: np.percentile(x.dropna(), 85),
        lambda x: np.percentile(x.dropna(), 90),
        lambda x: np.percentile(x.dropna(), 95)
    ]
}

df_stats_res = df_raw_emp.groupby(['SEMA_REF', 'SEGMENT', 'SPEED'], as_index=False).agg(agg_dict)

df_stats_res.columns = [
    'SEMA_REF' if col[0] == 'SEMA_REF' else
    'SEGMENT' if col[0] == 'SEGMENT' else
    'SPEED' if col[0] == 'SPEED' else
    f"CONSUMO_{col[1]}".replace('<lambda>', 'percentil')
    for col in df_stats_res.columns
]

df_stats_res = df_stats_res.rename(columns={
    'CONSUMO_<lambda_0>': 'CONSUMO_PERCENTIL_70',
    'CONSUMO_<lambda_1>': 'CONSUMO_PERCENTIL_75',
    'CONSUMO_<lambda_2>': 'CONSUMO_PERCENTIL_80',
    'CONSUMO_<lambda_3>': 'CONSUMO_PERCENTIL_85',
    'CONSUMO_<lambda_4>': 'CONSUMO_PERCENTIL_90',
    'CONSUMO_<lambda_5>': 'CONSUMO_PERCENTIL_95'
})

df_snowpark = session.create_dataframe(df_stats_res)

In [ ]:
#df_raw.count()

In [ ]:
from snowflake.snowpark.functions import current_timestamp

# Adicionar coluna com timestamp atual
df_snowpark = df_snowpark.with_column("DATA_INGESTAO", current_timestamp())

In [ ]:
from snowflake.snowpark.functions import col

tabela_destino = "DWDEV.DATASCIENCE.BEEGOL_PERCENTIS_TRAFEGO"

check_query = f"""
    SELECT COUNT(*) 
    FROM DWDEV.INFORMATION_SCHEMA.TABLES 
    WHERE TABLE_SCHEMA = 'DATASCIENCE' 
    AND TABLE_NAME = 'BEEGOL_PERCENTIS_TRAFEGO'
"""

resultado = session.sql(check_query).collect()
tabela_existe = resultado[0][0] > 0

if not tabela_existe:
    print(f"A tabela {tabela_destino} não existe. Criando e salvando dados...")

    df_snowpark.write.save_as_table(
        tabela_destino, 
        mode="overwrite"
    )
    print("Dados salvos com sucesso!")
else:
    print(f"A tabela {tabela_destino} já existe. Nenhuma ação realizada.")
    # Pegar semanas únicas que queremos inserir (já como string)
    unique_list = (
        df_snowpark
        .select(col("SEMA_REF").cast("string").alias("SEMA_REF"))
        .distinct()
        .to_pandas()["SEMA_REF"]
        .tolist()
    )
    
    if not unique_list:
        print("Nenhum dado para inserir.")
    else:
        # Pegar semanas que JÁ EXISTEM na tabela destino
        semanas_existentes = (
            session.table(tabela_destino)
            .select(col("SEMA_REF").cast("string").alias("SEMA_REF"))
            .filter(col("SEMA_REF").isin(unique_list))
            .distinct()
            .to_pandas()["SEMA_REF"]
            .tolist()
        )
    
        
        if not semanas_existentes:
            print("Nenhuma semana encontrada na tabela destino")
    
        else:
            semanas_para_inserir = [s for s in unique_list if s not in semanas_existentes]
            print(semanas_para_inserir)
            
            if not semanas_para_inserir:
                print("Todas as semanas já existem na tabela destino.")
                print("Nenhum dado será inserido.")
            else:
                print("Semanas que já existem:", semanas_existentes)
                print("Semanas que serão inseridas:", semanas_para_inserir)
    
                df_para_inserir = df_snowpark.filter(
                    col("SEMA_REF").cast("string").isin(semanas_para_inserir)
                )
                df_para_inserir
    
                print(f"Inserindo dados para {len(semanas_para_inserir)} semana(s) novas...")
                df_para_inserir.write.mode("append").save_as_table(tabela_destino)


In [ ]:
#df_snowpark.count()